# Chavruta.AI — Stage 1: fetch the **commercially-licensed** corpus, per tier (Kaggle · CPU)

For every source in the Jewish bookshelf — a base text, a commentary, anything — this notebook:

1. asks Sefaria **which editions exist** for it,
2. keeps only editions whose licence **permits commercial (paid) use** — Public Domain, CC0, CC-BY,
   CC-BY-SA — and **picks exactly one** (the least legally-encumbered, preferring vocalized text),
3. fetches **only that edition**, recording its licence + edition on every chunk,
4. **skips** any source with no commercial edition (e.g. Steinsaltz, most of the Zohar),
5. uploads each tier to its **own new HF dataset repo**, automatically.

Runs on **CPU** — it is network-bound (Sefaria), not compute-bound. This is stage 1 of two; stage 2
embeds these chunks on a GPU (`scripts/ingest_job.py`).

**Not legal advice** — it encodes the licence terms Sefaria publishes per edition (verified live
2026-07-17). Get counsel before charging money; `sefaria.org/terms` still needs a human read.


## 1. Config — the cell you edit

In [ ]:
# One NEW HF dataset repo per tier: f"{HF_NAMESPACE}/{REPO_PREFIX}{slug}"
HF_NAMESPACE = "Yehuda-Rubin"
REPO_PREFIX  = "chavruta-commercial-"      # new repos, kept separate from the existing free-tier ones

# Tiers to build THIS run. Fetching everything is many hours — run a few per Kaggle session; each
# tier is resumable. Ordered smallest→largest so a short session still finishes something.
TIERS_TO_RUN = ["musar", "reference", "tosefta"]     # then: liturgy, kabbalah, midrash, chasidut,
#   jewish_thought, second_temple, shut, yerushalmi, mishnah, tanakh, halacha, gemara

PREFER_NIKKUD = True       # when several commercial editions exist, prefer a vocalized one
SLEEP = 0.2                # politeness pause between Sefaria calls

import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("repos    :", f"{HF_NAMESPACE}/{REPO_PREFIX}<slug>")
print("tiers    :", TIERS_TO_RUN)
print("work dir :", WORK)

In [ ]:
# slug → (Sefaria top category, optional keep-predicate on a work).
# The Talmud category holds BOTH Bavli and Yerushalmi, so they split on the title.
TIERS = {
    "tanakh":         ("Tanakh",         None),
    "mishnah":        ("Mishnah",        None),
    "gemara":         ("Talmud",         lambda w: "Jerusalem Talmud" not in w["title"]),
    "yerushalmi":     ("Talmud",         lambda w: "Jerusalem Talmud" in w["title"]),
    "halacha":        ("Halakhah",       None),
    "shut":           ("Responsa",       None),
    "midrash":        ("Midrash",        None),
    "kabbalah":       ("Kabbalah",       None),
    "chasidut":       ("Chasidut",       None),
    "musar":          ("Musar",          None),
    "jewish_thought": ("Jewish Thought", None),
    "tosefta":        ("Tosefta",        None),
    "liturgy":        ("Liturgy",        None),
    "second_temple":  ("Second Temple",  None),
    "reference":      ("Reference",      None),
}
assert all(t in TIERS for t in TIERS_TO_RUN), "unknown tier in TIERS_TO_RUN"

In [ ]:
!pip install -q "requests>=2.32" "huggingface_hub>=0.23"

## 2. Hugging Face login — needs a **write** token
Kaggle → Add-ons → Secrets → add `HF_TOKEN`. Or paste when prompted.

In [ ]:
from huggingface_hub import login, HfApi
tok = os.environ.get("HF_TOKEN")
if not tok:
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        import getpass; tok = getpass.getpass("HF write token: ")
login(token=tok); HF = HfApi()
print("logged in as:", HF.whoami()["name"])

## 3. Sefaria API — versions, TOC walk, and version-pinned fetch

In [ ]:
import json, re, time, urllib.parse
import requests

BASE = "https://www.sefaria.org"
S = requests.Session()
S.headers.update({"User-Agent": "Chavruta.AI/0.2 (educational Torah RAG; commercial-licence fetch)"})

def _get_json(path, params=None, retries=4):
    url = f"{BASE}/api/{path}"
    for a in range(retries):
        try:
            r = S.get(url, params=params, timeout=120)
        except requests.RequestException:
            time.sleep(2 * (a + 1)); continue
        if r.status_code == 404: return None
        if r.status_code == 200:
            try: return r.json()
            except Exception: return None
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(3 * (a + 1)); continue
        return None
    return None

_VERSIONS_CACHE = {}
def versions(title):
    """All editions Sefaria holds for a title (cached). [] if unknown/none."""
    if title not in _VERSIONS_CACHE:
        data = _get_json(f"texts/versions/{urllib.parse.quote(title)}")
        _VERSIONS_CACHE[title] = data if isinstance(data, list) else []
        time.sleep(0.05)
    return _VERSIONS_CACHE[title]

def fetch_text(ref, he_version=None, en_version=None, want_he=True, want_en=True):
    """→ (he_nested, en_nested, rights) or None. Only the requested languages/editions are fetched;
    `rights` carries the licence + edition the text actually came from, per language."""
    vparams = []
    if want_he: vparams.append(f"hebrew|{he_version}" if he_version else "hebrew")
    if want_en: vparams.append(f"english|{en_version}" if en_version else "english")
    if not vparams: return None
    data = _get_json(f"v3/texts/{urllib.parse.quote(ref)}",
                     params={"version": vparams, "return_format": "text_only"})
    if not data: return None
    he = en = None
    rights = {"he": {"license": "", "version_title": ""}, "en": {"license": "", "version_title": ""}}
    for v in data.get("versions", []):
        fam = (v.get("languageFamilyName") or v.get("language") or "").lower()
        meta = {"license": v.get("license") or "", "version_title": v.get("versionTitle") or ""}
        if fam.startswith("he") and he is None: he, rights["he"] = v.get("text"), meta
        elif fam.startswith("en") and en is None: en, rights["en"] = v.get("text"), meta
    return he, en, rights

# ── category TOC → every work in the tier (base texts AND commentaries) ──
PERIOD_OVERRIDE = {"Geonim": "geonim", "Rishonim": "rishonim", "Acharonim": "acharonim", "Modern": "modern"}
def slugify(cat): return re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")

_WORKS_CACHE = {}
def load_works(category):
    if category in _WORKS_CACHE: return _WORKS_CACHE[category]
    toc = _get_json("index/")
    root = next((c for c in toc if c.get("category") == category), None)
    if root is None:
        raise SystemExit(f"category {category!r} not found on Sefaria")
    default_period, works = slugify(category), []
    def walk(node, period, path):
        cat = node.get("category", "")
        period = PERIOD_OVERRIDE.get(cat, period)
        path = path + [cat] if cat else path
        for ch in node.get("contents", []):
            if "contents" in ch: walk(ch, period, path)
            elif ch.get("title"):
                works.append({"title": ch["title"], "he_title": ch.get("heTitle", ""),
                              "period": period, "category_path": " / ".join(path)})
    walk(root, default_period, [])
    _WORKS_CACHE[category] = works
    return works

def _node_titles(node):
    en = he = ""
    for t in node.get("titles", []):
        if t.get("primary"):
            if t.get("lang") == "en": en = t.get("text", "")
            elif t.get("lang") == "he": he = t.get("text", "")
    return en or node.get("key", ""), he

def leaf_ref_bases(title, he_title):
    idx = _get_json(f"v2/raw/index/{urllib.parse.quote(title)}")
    schema = (idx or {}).get("schema", {})
    out = []
    def walk(node, en_ref, he_ref, section_en):
        if "nodes" in node:
            for ch in node["nodes"]:
                en_t, he_t = _node_titles(ch)
                if ch.get("default") or not en_t: walk(ch, en_ref, he_ref, section_en)
                else: walk(ch, f"{en_ref}, {en_t}", f"{he_ref}, {he_t}" if he_t else he_ref,
                           en_t if not section_en else f"{section_en} / {en_t}")
        else:
            out.append((en_ref, he_ref, section_en))
    if "nodes" in schema: walk(schema, title, he_title, "")
    else: out.append((title, he_title, ""))
    return out

## 4. Licence rules — pick the ONE commercial edition per source

In [ ]:
_CC_NC = ("cc-by-nc", "cc by-nc", "cc-nc")
_GRANTS = {"public domain", "public domain mark", "pd", "cc0", "cc0 1.0", "cc zero",
           "cc-by", "cc by", "cc-by 4.0", "cc-by-sa", "cc by-sa", "cc-by-sa 4.0"}

def allows_commercial_use(lic):
    """Fail closed: True only for licences that positively grant commercial reproduction."""
    x = (lic or "").strip().lower()
    if x in {"", "unknown", "none", "null"}: return False
    if x.startswith(_CC_NC) or x.startswith("copyright"): return False
    return x in _GRANTS

def _lic_rank(lic):
    x = (lic or "").strip().lower()
    if x in {"public domain", "public domain mark", "pd", "cc0", "cc0 1.0", "cc zero"}: return 2  # no obligations
    if x.startswith(("cc-by", "cc by")) and "sa" not in x: return 1                                # attribution only
    return 0                                                                                       # cc-by-sa: + share-alike

def _has_nikkud(vt):
    return any(k in (vt or "") for k in ("Nikkud", "nikkud", "Vocaliz", "vocaliz", "Ta'amei", "Masorah"))

def pick_commercial(title, lang):
    """The single best commercial edition of `title` in `lang`, or None. Preference: least
    legally-encumbered licence, then (optionally) vocalized, then Sefaria's primary, then priority."""
    cands = [v for v in versions(title)
             if (v.get("language") or "").lower().startswith(lang) and allows_commercial_use(v.get("license"))]
    if not cands: return None
    best = max(cands, key=lambda v: (
        _lic_rank(v.get("license")),
        _has_nikkud(v.get("versionTitle")) if PREFER_NIKKUD else 0,
        1 if v.get("isPrimary") else 0,
        v.get("priority") or 0))
    return {"version_title": best.get("versionTitle") or "", "license": best.get("license") or ""}

from collections import Counter
KEPT, DROPPED, SKIPPED_WORKS = Counter(), Counter(), []

## 5. Chunk builder — carries the licence, asserts commercial-only

In [ ]:
def _seg(x):
    if isinstance(x, list): return " ".join(_seg(i) for i in x if i).strip()
    return (x or "").strip() if isinstance(x, str) else ""

def _walk_leaves(he, en, path):
    if isinstance(he, list):
        for i, sub in enumerate(he):
            en_sub = en[i] if isinstance(en, list) and i < len(en) else None
            yield from _walk_leaves(sub, en_sub, path + [i + 1])
    else:
        yield path, (he if isinstance(he, str) else ""), (en if isinstance(en, str) else "")

def build_chunks(work, slug, en_ref, section_en, he_nested, en_nested, rights):
    he_r, en_r = rights.get("he") or {}, rights.get("en") or {}
    he_lic, en_lic = he_r.get("license", ""), en_r.get("license", "")
    he_ok, en_ok = allows_commercial_use(he_lic), allows_commercial_use(en_lic)
    book, he_book = work["title"], (work["he_title"] or work["title"])
    out = []
    for path, he_t, en_t in _walk_leaves(he_nested, en_nested, []):
        he_t, en_t = _seg(he_t), _seg(en_t)
        keep_he = he_t if (he_ok and he_t) else ""
        keep_en = en_t if (en_ok and en_t) else ""
        if he_t and not he_ok: DROPPED[he_lic or "(unknown)"] += 1
        if en_t and not en_ok: DROPPED[en_lic or "(unknown)"] += 1
        if not (keep_he or keep_en): continue
        if keep_he: KEPT[he_lic] += 1
        if keep_en: KEPT[en_lic] += 1
        path_str = ".".join(str(p) for p in path) or "1"
        verse_id = f"{en_ref}.{path_str}".replace(" ", "_")
        ref_label = f"{en_ref} {':'.join(str(p) for p in path)}".strip()
        doc = f"[{he_book}] {ref_label}\n{keep_he}\n{keep_en}".strip()
        out.append({"id": f"{verse_id}_{slug}", "document": doc, "metadata": {
            "verse_id": verse_id, "ref": ref_label, "book": book,
            "chapter": path[0] if path else 1, "verse": path[-1] if path else 1,
            "chunk_type": slug, "commentator": "", "work": slug, "period": work["period"],
            "author_he": he_book, "section": section_en, "category_path": work["category_path"],
            "text_he": keep_he, "text_en": keep_en,
            "license_he": he_lic, "version_he": he_r.get("version_title", ""),
            "license_en": en_lic, "version_en": en_r.get("version_title", ""),
        }})
    return out

## 6. Tier runner — resumable; one JSONL per tier

In [ ]:
def build_tier(slug):
    category, keep = TIERS[slug]
    works = load_works(category)
    if keep: works = [w for w in works if keep(w)]
    out_jsonl = f"{WORK}/{slug}.jsonl"
    done_path = f"{WORK}/{slug}.done"
    done = set()
    if os.path.exists(done_path):
        done = {l.strip() for l in open(done_path, encoding="utf-8") if l.strip()}
    print(f"[{slug}] {category}: {len(works)} works, {len(done)} already done", flush=True)
    fout = open(out_jsonl, "a", encoding="utf-8")
    try:
        for i, work in enumerate(works, 1):
            title = work["title"]
            if title in done: continue
            # pick ONE commercial edition per language for this source
            he_sel, en_sel = pick_commercial(title, "he"), pick_commercial(title, "en")
            if not he_sel and not en_sel:
                SKIPPED_WORKS.append(title)
                with open(done_path, "a", encoding="utf-8") as df: df.write(title + "\n")
                if i % 25 == 0: print(f"  [{i}/{len(works)}] …", flush=True)
                continue
            n = 0
            for en_ref, he_ref, section_en in leaf_ref_bases(title, work["he_title"]):
                res = fetch_text(en_ref,
                                 he_version=he_sel["version_title"] if he_sel else None,
                                 en_version=en_sel["version_title"] if en_sel else None,
                                 want_he=bool(he_sel), want_en=bool(en_sel))
                if not res or not (res[0] or res[1]): time.sleep(0.15); continue
                for c in build_chunks(work, slug, en_ref, section_en, *res):
                    fout.write(json.dumps(c, ensure_ascii=False) + "\n"); n += 1
                time.sleep(SLEEP)
            fout.flush()
            with open(done_path, "a", encoding="utf-8") as df: df.write(title + "\n")
            tag = (he_sel or en_sel)["license"]
            print(f"  [{i}/{len(works)}] {title[:44]:44} {tag:12} → {n:,} chunks", flush=True)
    finally:
        fout.close()
    rows = [json.loads(l) for l in open(out_jsonl, encoding="utf-8") if l.strip()]
    print(f"  ✅ {slug}: {len(rows):,} chunks")
    return out_jsonl, len(rows)

## 7. Run the selected tiers (the long part; CPU is fine, resumable)

In [ ]:
BUILT = {}
for slug in TIERS_TO_RUN:
    path, n = build_tier(slug)
    if n: BUILT[slug] = path
print("\nbuilt:", {k: v for k, v in BUILT.items()})
print("skipped (no commercial edition):", len(SKIPPED_WORKS), "works")

## 8. Licence audit — verify before uploading

In [ ]:
print("KEPT (in the datasets):")
for lic, n in KEPT.most_common():
    print(f"   {'ok  ' if allows_commercial_use(lic) else '??  '}{lic:26} {n:,}")
print("\nDROPPED (non-commercial, excluded):")
for lic, n in DROPPED.most_common(12):
    print(f"   STOP {lic:26} {n:,}")
print(f"\nskipped whole works (no commercial edition at all): {len(SKIPPED_WORKS)}")
for t in SKIPPED_WORKS[:15]: print("     -", t)
assert all(allows_commercial_use(l) for l in KEPT), "a non-commercial licence reached KEPT"
print("\naudit OK — every kept licence grants commercial use")

## 9. Upload — each tier to its **own new** HF dataset repo, automatically

In [ ]:
from huggingface_hub import create_repo
for slug, path in BUILT.items():
    repo = f"{HF_NAMESPACE}/{REPO_PREFIX}{slug}"
    create_repo(repo, repo_type="dataset", exist_ok=True, token=tok)
    print(f"⬆️  {slug}.jsonl → {repo}", flush=True)
    HF.upload_file(path_or_fileobj=path, path_in_repo=f"{slug}.jsonl",
                   repo_id=repo, repo_type="dataset", token=tok)
print("\ndone. each tier is its own dataset. Stage 2 embeds them into the index.")

## Next — Stage 2 (GPU)
These per-tier datasets hold the licensed **chunks**. Stage 2 embeds them (bge-m3 on a GPU) and
publishes the index: `scripts/ingest_job.py`, then load with `scripts/load_all_indexes.py` +
`scripts/create_payload_indexes.py`. Every chunk already carries `license_he`/`license_en`, so the
served product can filter by rights and attribute the exact edition.